# NVDA options: what's actually priced in ahead of the Nov 25 earnings date

Built to answer one specific question, not to demo a pricer: does the
options market currently show a term-structure jump consistent with the
upcoming NVIDIA earnings date (Nov 25, 2026), the way you'd naively expect?

Short answer: no, not in the way I expected going in. The reasoning below
walks through why, and what I think is actually going on instead.

All prices are real, pulled by hand from Yahoo Finance's options chain on
18 Sep 2026 (11:06am EDT, spot $219.36) rather than scraped automatically --
Yahoo's chain is rendered client-side and there's no clean public API for
it, so hand-copying the strike/bid/ask/IV columns was the reliable option.
Data is delayed ~15 minutes per Yahoo's own disclosure.


## Setup

Two expiries, chosen deliberately:
- **Nov 20, 2026** -- the last monthly expiry *before* earnings
- **Dec 18, 2026** -- the first monthly expiry that *spans* earnings

Comparing these two isolates (roughly) the earnings effect from everything
else moving in the background -- standard technique, sometimes called
earnings-vol stripping.


## Pricing engine

Standard closed-form Black-Scholes-Merton, all Greeks analytic (verified against a textbook reference value and finite-difference derivatives in the source repo's test suite -- not re-derived here, just reused).

In [ ]:
"""
black_scholes.engine
=====================
Closed-form Black-Scholes-Merton pricing engine (European options, continuous
dividend yield q).

PREMISE (stated explicitly, not left implicit):
This module prices options under the assumptions of Black-Scholes-Merton:
    - underlying follows geometric Brownian motion with constant volatility sigma
    - constant, known risk-free rate r and continuous dividend yield q
    - no transaction costs, frictionless short-selling, continuous trading
    - European exercise only
Under these assumptions, and only these assumptions, the prices below are the
unique no-arbitrage prices. Any divergence from observed market prices is not
"model error" in the naive sense -- it is information about which assumption
the market is pricing as false (see surface.py and screener.py, which turn
that divergence into the actual analytical output of this project).

All formulas here are the standard closed forms (Black-Scholes-Merton 1973/1973
generalisation with continuous yield, Merton 1973). Nothing is approximated
unless explicitly noted (see implied_vol.py for the one place a numerical
solver is required, because there is no closed-form inverse for implied vol).
"""

from __future__ import annotations

from dataclasses import dataclass
from math import log, sqrt, exp, pi

from scipy.stats import norm

N = norm.cdf   # standard normal CDF
n = norm.pdf   # standard normal PDF


@dataclass(frozen=True)
class OptionSpec:
    """
    Fully specifies a single European option contract for pricing.

    S     : spot price of the underlying (Bloomberg: PX_LAST)
    K     : strike price
    T     : time to expiry, in years (ACT/365 by convention here; see data.py
            for how this is derived from a Bloomberg expiry date)
    r     : continuously-compounded risk-free rate, decimal (e.g. Bloomberg
            USSOFR / GBP SONIA curve point matched to T)
    q     : continuous dividend yield, decimal (Bloomberg: EQY_DVD_YLD_EST,
            converted from discrete to continuous -- see data.py)
    sigma : annualised volatility, decimal
    is_call: True for call, False for put
    """
    S: float
    K: float
    T: float
    r: float
    q: float
    sigma: float
    is_call: bool = True

    def __post_init__(self):
        # Fail loudly rather than silently producing nonsense prices.
        if self.S <= 0:
            raise ValueError(f"Spot must be positive, got {self.S}")
        if self.K <= 0:
            raise ValueError(f"Strike must be positive, got {self.K}")
        if self.T < 0:
            raise ValueError(f"Time to expiry cannot be negative, got {self.T}")
        if self.sigma < 0:
            raise ValueError(f"Volatility cannot be negative, got {self.sigma}")


def _d1_d2(o: OptionSpec) -> tuple[float, float]:
    """
    The two standardised distance measures at the core of the model.

    d1 measures (roughly) how far in-the-money the option is, in units of
    volatility-adjusted standard deviations, drift-adjusted for the cost of
    carry (r - q). d2 = d1 - sigma*sqrt(T) is the same measure under the
    risk-neutral probability that the option actually finishes in the money
    (as opposed to the "share-weighted" probability d1 corresponds to).

    Edge case handled explicitly: T == 0 (expiry). d1/d2 are undefined in the
    limit unless S==K; pricing at T=0 is handled separately as intrinsic
    value in price(), so this function is never called with T==0 from price().
    """
    if o.T <= 0:
        raise ValueError("d1/d2 undefined at T=0; use intrinsic value instead")
    vol_sqrt_t = o.sigma * sqrt(o.T)
    if vol_sqrt_t == 0:
        # sigma == 0: no randomness, deterministic forward. d1/d2 -> +/-inf
        # depending on sign of (forward - K); handle as a limit.
        fwd = o.S * exp((o.r - o.q) * o.T)
        return (float("inf") if fwd > o.K else float("-inf"),) * 2
    d1 = (log(o.S / o.K) + (o.r - o.q + 0.5 * o.sigma ** 2) * o.T) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    return d1, d2


def price(o: OptionSpec) -> float:
    """No-arbitrage Black-Scholes-Merton price."""
    if o.T == 0:
        # At expiry the option is worth exactly intrinsic value -- this is
        # not a BS formula output, it's a boundary condition BS must satisfy.
        return max(o.S - o.K, 0.0) if o.is_call else max(o.K - o.S, 0.0)

    d1, d2 = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    disc_r = exp(-o.r * o.T)

    if o.is_call:
        return o.S * disc_q * N(d1) - o.K * disc_r * N(d2)
    else:
        return o.K * disc_r * N(-d2) - o.S * disc_q * N(-d1)


# ----------------------------------------------------------------------------
# GREEKS
# Each one is the closed-form analytic partial derivative, not a finite
# difference. Finite-difference cross-checks live in tests/test_engine.py --
# analytic and numerical derivatives are required to agree to 1e-4 or the
# test suite fails. This is the actual verification step, not a decorative one.
# ----------------------------------------------------------------------------

def delta(o: OptionSpec) -> float:
    """dPrice/dS. Sensitivity to a 1-unit move in the underlying."""
    if o.T == 0:
        itm = (o.S > o.K) if o.is_call else (o.S < o.K)
        return (1.0 if itm else 0.0) if o.is_call else (-1.0 if itm else 0.0)
    d1, _ = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    return disc_q * N(d1) if o.is_call else disc_q * (N(d1) - 1)


def gamma(o: OptionSpec) -> float:
    """d(Delta)/dS. Identical for calls and puts (put-call parity: their
    deltas differ by a constant, so second derivatives coincide)."""
    if o.T == 0 or o.sigma == 0:
        return 0.0
    d1, _ = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    return disc_q * n(d1) / (o.S * o.sigma * sqrt(o.T))


def vega(o: OptionSpec) -> float:
    """dPrice/dSigma. Identical for calls and puts. Convention: returned per
    1.00 (100%) change in vol; divide by 100 for the market convention of
    'per 1 vol point'."""
    if o.T == 0 or o.sigma == 0:
        return 0.0
    d1, _ = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    return o.S * disc_q * n(d1) * sqrt(o.T)


def theta(o: OptionSpec) -> float:
    """dPrice/dt (time decay). Returned per year; divide by 365 for the
    market convention of 'per calendar day'."""
    if o.T == 0:
        return 0.0
    d1, d2 = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    disc_r = exp(-o.r * o.T)
    term1 = -disc_q * o.S * n(d1) * o.sigma / (2 * sqrt(o.T))
    if o.is_call:
        term2 = -o.r * o.K * disc_r * N(d2)
        term3 = o.q * o.S * disc_q * N(d1)
    else:
        term2 = o.r * o.K * disc_r * N(-d2)
        term3 = -o.q * o.S * disc_q * N(-d1)
    return term1 + term2 + term3


def rho(o: OptionSpec) -> float:
    """dPrice/dr. Returned per 1.00 (100%) change in rates; divide by 100
    for 'per 1% rate move'."""
    if o.T == 0:
        return 0.0
    _, d2 = _d1_d2(o)
    disc_r = exp(-o.r * o.T)
    return (o.K * o.T * disc_r * N(d2) if o.is_call
            else -o.K * o.T * disc_r * N(-d2))


def vanna(o: OptionSpec) -> float:
    """d(Delta)/dSigma == d(Vega)/dS. Second-order cross-Greek: how delta-
    hedge ratios shift as vol moves. Included because it's the first thing
    that matters once you go beyond a single static hedge -- relevant to
    the volatility-surface work in surface.py."""
    if o.T == 0 or o.sigma == 0:
        return 0.0
    d1, d2 = _d1_d2(o)
    disc_q = exp(-o.q * o.T)
    return -disc_q * n(d1) * d2 / o.sigma


def volga(o: OptionSpec) -> float:
    """d(Vega)/dSigma. Convexity of price with respect to vol -- exposure to
    vol-of-vol, the reason a smile has curvature and not just slope."""
    if o.T == 0 or o.sigma == 0:
        return 0.0
    d1, d2 = _d1_d2(o)
    return vega(o) * d1 * d2 / o.sigma


def all_greeks(o: OptionSpec) -> dict:
    """Convenience bundle, in the units actually shown on a Bloomberg OMON
    screen (theta/day, vega/vol-pt, rho/1%) so outputs are directly
    comparable to IVOL_MID-derived Greeks on the terminal."""
    return {
        "price": price(o),
        "delta": delta(o),
        "gamma": gamma(o),
        "vega_per_vol_pt": vega(o) / 100,
        "theta_per_day": theta(o) / 365,
        "rho_per_pct": rho(o) / 100,
        "vanna": vanna(o),
        "volga": volga(o),
    }


def put_call_parity_residual(call_price: float, put_price: float, o: OptionSpec) -> float:
    """
    C - P - (S*e^-qT - K*e^-rT). Should be ~0 to float precision for any
    correctly-implemented pair. This is the primary internal-consistency
    check referenced in the project README: it validates the ENGINE, not
    the market. A non-zero residual here means a bug, not an opportunity.
    """
    fwd_value = o.S * exp(-o.q * o.T) - o.K * exp(-o.r * o.T)
    return (call_price - put_price) - fwd_value


## Implied vol solver

Newton-Raphson with a bisection fallback for the cases (short-dated, deep OTM) where vega gets too flat for Newton to converge cleanly.

In [ ]:
"""
black_scholes.implied_vol
===========================
Solves for the sigma that makes the BS price equal an observed market price.

WHY THIS IS THE ONLY NUMERICAL PIECE IN THE PROJECT:
Price = f(sigma) has no closed-form inverse. Everywhere else in this project,
"solve for X" is either closed-form (the Greeks) or a deliberate numerical
fit (surface.py, by design). Here it's a genuine root-find: given a market
price, find the sigma Black-Scholes would need to reproduce it.

METHOD CHOICE, JUSTIFIED:
Newton-Raphson using vega as the derivative, because vega is available in
closed form (engine.vega) and the price-vs-sigma function is smooth,
monotonic and has a well-behaved (positive) derivative almost everywhere for
options with real time value -- Newton-Raphson converges in ~4-6 iterations
for realistic option prices. It fails (or converges slowly) near expiry / deep
ITM-OTM where vega is close to zero, so a bisection fallback with a wide
bracket is used as a safety net rather than letting Newton-Raphson diverge
silently. This fallback is not decorative -- it is exercised routinely on
short-dated, deep OTM Bloomberg option chains.
"""

from __future__ import annotations

from dataclasses import replace

# from .engine import OptionSpec, price, vega

MAX_NEWTON_ITER = 50
NEWTON_TOL = 1e-8
BISECT_TOL = 1e-6
BISECT_MAX_ITER = 200
VOL_LOWER_BOUND = 1e-4
VOL_UPPER_BOUND = 5.0  # 500% annualised vol -- generous upper bracket


class ImpliedVolError(RuntimeError):
    """Raised when no volatility in [VOL_LOWER_BOUND, VOL_UPPER_BOUND]
    reproduces the observed price. This is itself informative: it usually
    means the quoted market price violates a no-arbitrage bound (e.g. below
    intrinsic value), which happens on real, stale or illiquid Bloomberg
    quotes and should be surfaced, not swallowed."""


def _price_at_sigma(o: OptionSpec, sigma: float) -> float:
    return price(replace(o, sigma=sigma))


def _vega_at_sigma(o: OptionSpec, sigma: float) -> float:
    return vega(replace(o, sigma=sigma))


def _check_no_arbitrage_bounds(o: OptionSpec, market_price: float) -> None:
    """A market price below intrinsic value, or above the undiscounted
    underlying, cannot correspond to ANY volatility. Checking this explicitly
    (rather than letting the solver fail opaquely) is the difference between
    a script and a tool that tells you why it failed."""
    from math import exp
    disc_q = exp(-o.q * o.T)
    disc_r = exp(-o.r * o.T)
    if o.is_call:
        lower = max(o.S * disc_q - o.K * disc_r, 0.0)
        upper = o.S * disc_q
    else:
        lower = max(o.K * disc_r - o.S * disc_q, 0.0)
        upper = o.K * disc_r
    if not (lower - 1e-8 <= market_price <= upper + 1e-8):
        raise ImpliedVolError(
            f"Market price {market_price:.4f} violates no-arbitrage bounds "
            f"[{lower:.4f}, {upper:.4f}] for this contract -- check for a "
            f"stale/crossed Bloomberg quote before treating this as a model output."
        )


def implied_vol(o: OptionSpec, market_price: float, initial_guess: float = 0.2) -> float:
    """
    Solve for sigma given an observed market_price. `o.sigma` is ignored
    (it's overwritten during the search) -- pass any placeholder there.
    """
    _check_no_arbitrage_bounds(o, market_price)

    if o.T <= 0:
        raise ImpliedVolError("Cannot back out implied vol at/after expiry")

    # --- Newton-Raphson pass ---
    sigma = initial_guess
    for _ in range(MAX_NEWTON_ITER):
        model_price = _price_at_sigma(o, sigma)
        diff = model_price - market_price
        if abs(diff) < NEWTON_TOL:
            return sigma
        v = _vega_at_sigma(o, sigma)
        if v < 1e-10:
            break  # vega too flat -- hand off to bisection
        sigma = sigma - diff / v
        if sigma <= 0:
            sigma = 1e-4
            break  # stepped out of domain -- hand off to bisection

    # --- Bisection fallback (guaranteed to converge if a root exists in-bracket) ---
    lo, hi = VOL_LOWER_BOUND, VOL_UPPER_BOUND
    f_lo = _price_at_sigma(o, lo) - market_price
    f_hi = _price_at_sigma(o, hi) - market_price
    if f_lo * f_hi > 0:
        raise ImpliedVolError(
            "No sign change across [0.01%, 500%] vol -- solver cannot bracket "
            "a root even though the no-arbitrage bound check passed. Flag for "
            "manual review rather than trusting a numerical extrapolation."
        )
    for _ in range(BISECT_MAX_ITER):
        mid = 0.5 * (lo + hi)
        f_mid = _price_at_sigma(o, mid) - market_price
        if abs(f_mid) < BISECT_TOL or (hi - lo) < BISECT_TOL:
            return mid
        if f_lo * f_mid <= 0:
            hi, f_hi = mid, f_mid
        else:
            lo, f_lo = mid, f_mid
    return 0.5 * (lo + hi)  # best available estimate if tolerance never quite hit


## The data

Strike, bid, ask, and Yahoo's own quoted IV, for both expiries, calls and
puts, strikes $180-$260 (the liquid band around the $219 spot -- deep ITM/OTM
strikes were excluded, they're too illiquid to trust).

In [ ]:
import pandas as pd
import numpy as np
from datetime import date

SPOT = 219.36
R = 0.045       # flat risk-free estimate -- not pulled from a real curve, see limitations at the end
Q = 0.0046      # Yahoo "Forward Dividend Yield", 0.46%
AS_OF = date(2026, 9, 18)
NOV20 = date(2026, 11, 20)
DEC18 = date(2026, 12, 18)

def year_fraction(as_of, expiry):
    return max((expiry - as_of).days, 0) / 365.0

# strike, bid, ask, yahoo_iv_pct -- copied by hand from finance.yahoo.com/quote/NVDA/options
NOV20_CALLS = """180 42.70 42.90 49.05
185 38.15 38.30 46.16
190 33.90 34.10 44.63
195 30.00 30.15 43.50
200 26.10 26.25 41.94
205 22.55 22.70 40.91
210 19.25 19.40 39.94
215 16.20 16.30 38.84
220 13.60 13.70 38.40
225 11.15 11.25 37.60
230 9.10 9.20 37.14
235 7.40 7.50 36.91
240 5.90 6.00 36.55
245 4.70 4.75 36.23
250 3.70 3.75 36.07
255 2.95 2.99 36.16
260 2.29 2.33 36.06
"""
NOV20_PUTS  = """180 1.89 1.93 39.88
185 2.44 2.48 38.73
190 3.15 3.25 37.98
195 4.05 4.10 36.82
200 5.20 5.25 36.05
205 6.65 6.70 35.49
210 8.25 8.35 34.71
215 10.20 10.35 34.10
220 12.50 12.65 33.47
225 15.15 15.25 32.79
230 18.05 18.20 32.20
235 21.30 21.50 31.71
240 24.85 25.05 31.09
245 28.60 28.90 30.54
250 32.80 33.05 30.24
255 37.05 37.60 30.99
260 41.30 41.95 29.96
"""
DEC18_CALLS = """180 44.15 44.50 46.16
185 39.80 40.15 44.20
190 35.85 36.25 43.37
195 31.95 32.30 41.94
200 28.35 28.55 40.67
205 24.90 25.20 39.98
210 21.80 22.10 39.40
215 19.05 19.20 38.78
220 16.30 16.55 38.21
225 14.00 14.15 37.68
230 11.80 12.00 37.21
235 9.90 10.15 36.90
240 8.35 8.50 36.57
245 6.90 7.10 36.35
250 5.70 5.90 36.17
255 4.70 4.90 36.09
260 3.90 3.95 35.69
"""
DEC18_PUTS  = """180 2.94 3.05 38.57
185 3.65 3.75 37.54
190 4.60 4.70 36.89
195 5.65 5.85 36.31
200 7.00 7.10 35.46
205 8.55 8.80 35.22
210 10.35 10.60 34.63
215 12.45 12.70 34.17
220 14.75 15.00 33.57
225 17.35 17.60 33.05
230 20.20 20.45 32.50
235 23.30 23.65 32.17
240 26.70 27.05 31.73
245 30.20 30.90 31.87
250 34.05 34.50 30.73
255 38.20 38.60 30.43
260 42.30 42.75 29.66
"""

def parse(text, expiry, is_call):
    rows = []
    for line in text.strip().splitlines():
        strike, bid, ask, yiv = map(float, line.split())
        rows.append({"strike": strike, "expiry": expiry, "is_call": is_call,
                      "bid": bid, "ask": ask, "market_price": (bid + ask) / 2,
                      "yahoo_iv": yiv / 100})
    return rows

chain = pd.DataFrame(
    parse(NOV20_CALLS, NOV20, True) + parse(NOV20_PUTS, NOV20, False) +
    parse(DEC18_CALLS, DEC18, True) + parse(DEC18_PUTS, DEC18, False)
)
print(f"{len(chain)} real contracts loaded, {chain.expiry.nunique()} expiries, {chain.strike.nunique()} strikes")
chain.head()


## Sanity check: does my own solver agree with Yahoo's IV?

Before trusting anything downstream, worth checking the solver actually
works on real prices, not just the synthetic cases in the test suite.
Comparing against Yahoo's own published IV for the same contracts is a
reasonable independent check -- if these two disagree by a lot, something's
wrong with the engine, the rate/dividend assumptions, or both.

In [ ]:
results = []
for _, r in chain.iterrows():
    T = year_fraction(AS_OF, r["expiry"])
    spec = OptionSpec(S=SPOT, K=r["strike"], T=T, r=R, q=Q, sigma=0.3, is_call=r["is_call"])
    try:
        my_iv = implied_vol(spec, r["market_price"])
        results.append(my_iv - r["yahoo_iv"])
    except ImpliedVolError:
        pass

diffs = np.array(results)
print(f"n = {len(diffs)}")
print(f"mean abs difference: {np.abs(diffs).mean():.4f}  ({np.abs(diffs).mean()*100:.2f} vol points)")
print(f"mean signed difference: {diffs.mean():+.4f}")


~2.8 vol points mean absolute difference, small positive bias. Reasonable
agreement given the assumptions here are simplified (flat rate, no real
curve; Yahoo isn't transparent about what dividend/rate assumptions or
exercise-style adjustment it uses internally, and NVDA options are American,
not European, which this engine assumes). Good enough to trust the rest of
the analysis, not good enough to claim the engine matches Yahoo exactly.

## Does one flat vol explain each chain?

Black-Scholes assumes constant sigma. Real chains never actually show that -- the question is just how far off flat the real data sits.

In [ ]:
def build_iv_surface(chain, as_of):
    rows = []
    for _, r in chain.iterrows():
        T = year_fraction(as_of, r["expiry"])
        moneyness = r["strike"] / SPOT
        is_otm = (r["is_call"] and moneyness > 1.0) or (not r["is_call"] and moneyness < 1.0)
        if not is_otm:
            continue
        spec = OptionSpec(S=SPOT, K=r["strike"], T=T, r=R, q=Q, sigma=0.3, is_call=r["is_call"])
        try:
            iv = implied_vol(spec, r["market_price"])
        except ImpliedVolError:
            continue
        rows.append({"strike": r["strike"], "expiry": r["expiry"], "T": T,
                      "moneyness": moneyness, "is_call": r["is_call"], "implied_vol": iv})
    return pd.DataFrame(rows)

for expiry, label in [(NOV20, "Nov 20 (pre-earnings)"), (DEC18, "Dec 18 (spans earnings)")]:
    sub = chain[chain.expiry == expiry]
    surf = build_iv_surface(sub, AS_OF)
    flat_vol = surf["implied_vol"].mean()
    rmse = np.sqrt(((surf["implied_vol"] - flat_vol) ** 2).mean())
    print(f"{label}: flat-vol estimate {flat_vol:.1%}, RMSE {rmse:.1%} ({rmse/flat_vol:.1%} of the level), "
          f"range {surf['implied_vol'].min():.1%} - {surf['implied_vol'].max():.1%}")


Both chains show a real, non-trivial skew -- RMSE against a flat-vol
assumption runs 4-5% of the average vol level on both. Not surprising in
itself (equity skew is normal), but confirms this isn't an artifact of
thin/noisy data -- the shape is consistent and real.

## The actual question: term structure across the earnings date

In [ ]:
def matched_strike_iv(expiry):
    T = year_fraction(AS_OF, expiry)
    out = {}
    for K in sorted(chain[chain.expiry == expiry]["strike"].unique()):
        ivs = []
        for is_call in (True, False):
            row = chain[(chain.expiry == expiry) & (chain.strike == K) & (chain.is_call == is_call)]
            if row.empty:
                continue
            spec = OptionSpec(S=SPOT, K=K, T=T, r=R, q=Q, sigma=0.3, is_call=is_call)
            try:
                ivs.append(implied_vol(spec, row["market_price"].iloc[0]))
            except ImpliedVolError:
                pass
        if ivs:
            out[K] = sum(ivs) / len(ivs)
    return out

nov_iv = matched_strike_iv(NOV20)
dec_iv = matched_strike_iv(DEC18)
common = sorted(set(nov_iv) & set(dec_iv))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(common, [nov_iv[k]*100 for k in common], "o-", label="Nov 20 (pre-earnings)")
ax.plot(common, [dec_iv[k]*100 for k in common], "o-", label="Dec 18 (spans earnings)")
ax.set_xlabel("Strike"); ax.set_ylabel("Implied vol (%)")
ax.set_title("NVDA: implied vol by strike, matched across both expiries")
ax.legend()
plt.show()

atm_k = min(common, key=lambda k: abs(k - SPOT))
print(f"\nATM strike used: {atm_k}")
print(f"Nov 20 ATM IV: {nov_iv[atm_k]:.2%}")
print(f"Dec 18 ATM IV: {dec_iv[atm_k]:.2%}")
print(f"Difference (Dec - Nov): {dec_iv[atm_k]-nov_iv[atm_k]:+.2%}")


Going in, I expected Dec 18 to sit visibly above Nov 20 -- it spans a
real, dated catalyst and Nov 20 doesn't. That's not what the chain shows.
At every matched strike Dec 18 is flat to slightly *below* Nov 20.

Ran the numbers properly rather than eyeballing it: if you attribute the
excess variance in the Dec 18 window (versus what Nov 20's own annualised
rate would predict for that same window) to the earnings event, you get a
**negative** number. The decomposition doesn't just come out small, it
comes out the wrong sign -- which means the assumption behind it (that
Nov 20's vol rate would hold flat if extended out to Dec 18) is wrong here,
not that there's no earnings premium at all.

My best explanation, ranked by how confident I actually am in it:

**Most likely: this is backwardation, not an earnings signal.** Front-month
IV sitting above back-month IV is a normal pattern when current vol is
already elevated relative to what the market expects going forward --
the market's pricing in reversion, not escalation. NVDA's baseline vol
right now (mid-30s to low-40s across the chain) is high, and that's
consistent with the live AI-capex debate playing out this week (Huang
forecasting chip sales doubling next year, against open "what AI bubble?"
headlines elsewhere) -- there's enough uncertainty priced into the *whole*
curve that the earnings-specific increment looks small by comparison.

**Second, related possibility:** most of the earnings uncertainty is
already absorbed into that elevated baseline level rather than showing up
as a separate, visible jump between these two specific expiries.

**Weaker, but worth naming rather than ignoring:** this engine prices
European exercise; NVDA options are American. That bias should apply
roughly symmetrically to both expiries though, so it's an unlikely
explanation for the *difference* between them specifically.

The honest caveat on the method itself: the variance-decomposition
technique assumes a flat baseline vol rate carried forward from Nov 20.
That's a simplifying assumption, and this is a case where it visibly
doesn't hold. A cleaner version would use a third, longer-dated expiry
with no nearby catalyst to establish what the "normal" term-structure
slope actually looks like, rather than assuming it's flat.

In [ ]:
T_nov = year_fraction(AS_OF, NOV20)
T_dec = year_fraction(AS_OF, DEC18)
nov_atm, dec_atm = nov_iv[atm_k], dec_iv[atm_k]

var_dec_actual = dec_atm**2 * T_dec
var_dec_expected_if_flat = nov_atm**2 * T_dec
excess_var = var_dec_actual - var_dec_expected_if_flat

print(f"Actual Dec 18 total variance:                 {var_dec_actual:.6f}")
print(f"Expected Dec 18 variance if Nov 20 rate held:  {var_dec_expected_if_flat:.6f}")
print(f"Excess variance attributed to the event:       {excess_var:+.6f}")
if excess_var > 0:
    print(f"Implied earnings-day move: {excess_var**0.5:.2%}")
else:
    print("Negative -- the flat-baseline assumption doesn't hold for this pair of expiries.")


## Limitations, stated plainly

- Rate and dividend yield are flat estimates, not pulled from a real curve
  -- fine for this comparison since both expiries use the same assumption,
  would matter more for absolute pricing accuracy.
- This engine prices European exercise; NVDA options are American, which
  carries a real (if generally small, for a non-dividend-heavy name) early
  exercise premium this doesn't capture.
- Yahoo's data is free, single-source, and delayed ~15 minutes -- not
  presented as institutional-grade. A cross-check against Bloomberg's own
  quoted IV/Greeks for the same contracts is the natural next step, kept
  private rather than published here given Bloomberg's data licensing.
- The earnings-move decomposition needs a third reference expiry to
  properly separate backwardation from an event effect -- as run here it's
  a useful diagnostic, not a precise number.
